# 1D J1J2J3 inference: EuclideanRNN (seed 111-555)

This is part of the work arxiv:2604.24337 [quant-ph] by HL Dao. For the purpose of reproducing the results, the paths to the trained weights have to be adjusted to match the repo structure.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.append('../../1_hypnqs_torch_poincare/utility_poincare')
from j1j2j3_hyprnn_train_loop import *
numsamples = 10000

GPU Available:  False


In [2]:
def set_cpu_deterministic(seed):
    # 1. Python & Numpy
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 2. PyTorch CPU
    torch.manual_seed(seed)
    
    # 3. Force Deterministic Algorithms
    # This prevents non-deterministic CPU operations (like some views/reductions)
    torch.use_deterministic_algorithms(True)
    
    # 4. Limit CPU Threads
    # Setting this to 1 ensures operations are done in a fixed order.
    torch.set_num_threads(1)

In [3]:
def clip_local_energies(eloc, threshold=5.0):
    # Convert to numpy if it's a torch tensor, or vice versa
    eloc_real = np.real(eloc)
    median = np.median(eloc_real)
    mad = np.median(np.abs(eloc_real - median))    
    # Standard safety check to avoid division by zero if MAD is 0
    if mad == 0:
        return eloc        
    lower_bound = median - threshold * mad
    upper_bound = median + threshold * mad    
    # Clip the values (keeping the imaginary part if it exists)
    # Create a copy to avoid modifying the original array in place
    clipped = np.clip(eloc_real, lower_bound, upper_bound)    
    # If the original was complex, restore the imaginary part
    if np.iscomplexobj(eloc):
        return clipped + 1j * np.imag(eloc)
    return clipped

def define_load_test(wf,  numsamples,path_to_weights, Ee, clipped_e = False):
    state_dict = torch.load(path_to_weights, map_location=torch.device('cpu'))   
    new_state_dict = {}
    for key, value in state_dict.items():
        # Strip prefixes and rename keys to match current architecture
        new_key = key.replace('model.', '').replace('cell.', 'rnn.')
        new_state_dict[new_key] = value
    # This line loads the RE-MAPPED weights
    wf.model.load_state_dict(new_state_dict, strict=False)
    print("Successfully remapped and loaded weights.")
    
    # --- PART C: Check performance AFTER loading ---
    with torch.no_grad():
        test_samples_after = wf.sample(numsamples)
        if clipped_e:
            # 1. Get raw energies
            raw_gs_after = J1J2J3_local_energies(wf, N, J1, J2,J3, Bz, numsamples, test_samples_after, Marshall_sign=True)
    
            # 2. APPLY CLIPPING
            test_gs_after = clip_local_energies(raw_gs_after, threshold=5.0)
    
            # 3. Calculate statistics on cleaned data
            gs_mean_a = np.round(np.mean(test_gs_after), 4)
            gs_var_a = np.round(np.var(test_gs_after), 4)
    
            # Optional: Count how many were clipped to see if the model is unstable
            num_clipped = np.sum(np.real(raw_gs_after) != np.real(test_gs_after))
            print(f"Clipped {num_clipped} outlier samples out of {numsamples}")
        else:
            test_gs_after =  J1J2J3_local_energies(wf, N, J1, J2, J3, Bz, numsamples, test_samples_after, Marshall_sign=True)
            gs_mean_a = np.round(np.mean(test_gs_after),4)
            gs_var_a = np.round(np.var(test_gs_after),4)
    
    #wf.model.summary()
    #print('====================================================================')
    print(f'After loading weights, the ground state energy mean and variance are:')
    print(f'Mean E = {gs_mean_a}, var E = {gs_var_a}')
    print(f'DMRG energy  is {np.round(Ee,4)}')

In [4]:
N=100
syssize = 100
J1_ = 1.0
J1 = +J1_ * np.ones(syssize)
Bz = +0.0 * np.ones(syssize)
fname=f'results_EuclideanRNN'

## J2=0.0, J3=0.5

In [5]:
J2_ = 0.0
J3_ = 0.5
J2 = +J2_ * np.ones(syssize)
J3 = +J3_ * np.ones(syssize)
Ee_00_05= -53.99140745384424

In [7]:
seed=111
set_cpu_deterministic(seed)
units = 80
eRNN = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.0|J3=0.5_EuclRNN_80_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN, numsamples, e_wl, Ee_00_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.52109909057617+0.0005000000237487257j), var E = 0.01140000019222498
DMRG energy  is -53.9914
Time taken=0.117 hrs


In [16]:
seed=222
units = 80
eRNN = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.0|J3=0.5_EuclRNN_80_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN, numsamples, e_wl, Ee_00_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.612998962402344+0.0017999999690800905j), var E = 0.04879999905824661
DMRG energy  is -53.9914
Time taken=0.078 hrs


In [17]:
seed=333
units = 80
eRNN = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.0|J3=0.5_EuclRNN_80_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN, numsamples, e_wl, Ee_00_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.49660110473633-0.002099999925121665j), var E = 0.07479999959468842
DMRG energy  is -53.9914
Time taken=0.084 hrs


In [18]:
seed=444
units = 80
eRNN = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.0|J3=0.5_EuclRNN_80_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN, numsamples, e_wl, Ee_00_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-35.58729934692383+0.000699999975040555j), var E = 0.03280000016093254
DMRG energy  is -53.9914
Time taken=0.086 hrs


In [19]:
seed=555
units = 80
eRNN = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.0|J3=0.5_EuclRNN_80_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN, numsamples, e_wl, Ee_00_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.519901275634766+9.999999747378752e-05j), var E = 0.013100000098347664
DMRG energy  is -53.9914
Time taken=0.09 hrs


## J2=0.2, J3=0.2

In [8]:
J2_ = 0.2
J3_ = 0.2
J2 = +J2_ * np.ones(syssize)
J3 = +J3_ * np.ones(syssize)
Ee_02_02=-43.58595579948936

In [10]:
seed=111
set_cpu_deterministic(seed)
units = 80
eRNN = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2={J2_}|J3={J3_}_EuclRNN_80_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN, numsamples, e_wl, Ee_02_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-25.333900451660156+0.0007999999797903001j), var E = 0.05620000138878822
DMRG energy  is -43.586
Time taken=0.115 hrs


In [21]:
seed=222
units = 80
eRNN = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.2_EuclRNN_80_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN, numsamples, e_wl, Ee_02_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-25.365400314331055+0.0006000000284984708j), var E = 0.008299999870359898
DMRG energy  is -43.586
Time taken=0.075 hrs


In [22]:
seed=333
units = 80
eRNN = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.2_EuclRNN_80_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN, numsamples, e_wl, Ee_02_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-25.149900436401367+0.0010999999940395355j), var E = 0.10769999772310257
DMRG energy  is -43.586
Time taken=0.076 hrs


In [23]:
seed=444
units = 80
eRNN = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.2_EuclRNN_80_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN, numsamples, e_wl, Ee_02_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-25.223600387573242+0j), var E = 0.19329999387264252
DMRG energy  is -43.586
Time taken=0.076 hrs


In [24]:
seed=555
units = 80
eRNN = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.2_EuclRNN_80_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN, numsamples, e_wl, Ee_02_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-25.153200149536133-0.006000000052154064j), var E = 1.2662999629974365
DMRG energy  is -43.586
Time taken=0.079 hrs


## J2=0.2, J3 = 0.5

In [11]:
J2_ = 0.2
J3_ = 0.5
J2 = +J2_ * np.ones(syssize)
J3 = +J3_ * np.ones(syssize)
Ee_02_05=-49.628675088072384

In [12]:
seed=111
set_cpu_deterministic(seed)
units = 80
eRNN = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2={J2_}|J3={J3_}_EuclRNN_80_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN, numsamples, e_wl, Ee_02_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-32.527000427246094-0.00019999999494757503j), var E = 0.0035000001080334187
DMRG energy  is -49.6287
Time taken=0.111 hrs


In [27]:
seed=222
units = 80
eRNN = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.5_EuclRNN_80_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN, numsamples, e_wl, Ee_02_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-32.638301849365234-0.0005000000237487257j), var E = 0.008500000461935997
DMRG energy  is -49.6287
Time taken=0.082 hrs


In [28]:
seed=333
units = 80
eRNN = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.5_EuclRNN_80_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN, numsamples, e_wl, Ee_02_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-32.52730178833008-0j), var E = 0.004000000189989805
DMRG energy  is -49.6287
Time taken=0.077 hrs


In [29]:
seed=444
units = 80
eRNN = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.5_EuclRNN_80_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN, numsamples, e_wl, Ee_02_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-31.000699996948242+0j), var E = 0.0017999999690800905
DMRG energy  is -49.6287
Time taken=0.077 hrs


In [30]:
seed=555
units = 80
eRNN = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.5_EuclRNN_80_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN, numsamples, e_wl, Ee_02_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-32.671600341796875-0.003700000001117587j), var E = 0.2694999873638153
DMRG energy  is -49.6287
Time taken=0.076 hrs


## J2=0.5, J3=0.2

In [13]:
J2_ = 0.5
J3_= 0.2
J2 = +J2_ * np.ones(syssize)
J3 = +J3_ * np.ones(syssize)
Ee_05_02= -38.54733450537742

In [14]:
seed=111
set_cpu_deterministic(seed)
units = 80
eRNN = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2={J2_}|J3={J3_}_EuclRNN_80_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN, numsamples, e_wl, Ee_05_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-17.503000259399414+0.0008999999845400453j), var E = 0.008500000461935997
DMRG energy  is -38.5473
Time taken=0.115 hrs


In [32]:
seed=222
units = 80
eRNN = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.5|J3=0.2_EuclRNN_80_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN, numsamples, e_wl, Ee_05_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-17.5-0.00019999999494757503j), var E = 0.009700000286102295
DMRG energy  is -38.5473
Time taken=0.084 hrs


In [33]:
seed=333
units = 80
eRNN = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.5|J3=0.2_EuclRNN_80_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN, numsamples, e_wl, Ee_05_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-17.49169921875+0.0005000000237487257j), var E = 0.3334999978542328
DMRG energy  is -38.5473
Time taken=0.069 hrs


In [34]:
seed=444
units = 80
eRNN = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.5|J3=0.2_EuclRNN_80_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN, numsamples, e_wl, Ee_05_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-17.851299285888672+0.00039999998989515007j), var E = 0.10170000046491623
DMRG energy  is -38.5473
Time taken=0.075 hrs


In [35]:
seed=555
units = 80
eRNN = RNNwavefunction(systemsize=syssize, cell_type='EuclRNN', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.5|J3=0.2_EuclRNN_80_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eRNN, numsamples, e_wl, Ee_05_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-18.00429916381836+0.0027000000700354576j), var E = 0.015699999406933784
DMRG energy  is -38.5473
Time taken=0.076 hrs
